# Day 3 · Deliver & Govern — Lab 2
# Responsible AI & Governance Deep Dive · Operating Agents at Scale

Standard-library only, fully runnable top to bottom.

**Contents**
1. Responsible AI policy-as-code — a rule engine that checks an agent config against RAI policies
2. PII/PCI redaction guardrail — pattern-based pre-processing filter (demo-grade, not production regex)
3. Human-in-the-loop approval gate — decision router by risk/confidence
4. Audit trail — immutable-style append-only decision log
5. Operating at Scale — capacity planner (agents vs. throughput vs. cost)
6. Operating at Scale — SLO tracker + error-budget burn calculator
7. Multi-tenant agent registry — governance metadata per deployed agent instance


In [ ]:
import re, json, time, uuid, hashlib
from dataclasses import dataclass, field
from typing import Dict, Any, List, Optional
from datetime import datetime, timedelta, timezone

print("Ready.")


## 1. Responsible AI policy-as-code

Rather than a static PDF policy, express RAI requirements as **checkable rules** against
an agent's declared configuration. This lets governance run automatically at every deploy,
not just at annual review time.

In [ ]:
@dataclass
class AgentConfig:
    name: str
    has_human_review_for_high_risk: bool
    logs_decisions: bool
    pii_redaction_enabled: bool
    max_autonomy_level: str          # "advisory" | "approval_required" | "autonomous"
    data_residency: str              # e.g. "US", "EU"
    model_provider: str
    fallback_provider: Optional[str] = None
    bias_testing_completed: bool = False


@dataclass
class PolicyRule:
    rule_id: str
    description: str
    check: Any    # Callable[[AgentConfig], bool]
    severity: str  # "blocking" | "warning"


POLICY_RULES: List[PolicyRule] = [
    PolicyRule("RAI-01", "High-autonomy agents must have human review for high-risk actions",
               lambda c: c.max_autonomy_level != "autonomous" or c.has_human_review_for_high_risk,
               "blocking"),
    PolicyRule("RAI-02", "All agent decisions must be logged for auditability",
               lambda c: c.logs_decisions, "blocking"),
    PolicyRule("RAI-03", "PII/PCI redaction must be enabled before external model calls",
               lambda c: c.pii_redaction_enabled, "blocking"),
    PolicyRule("RAI-04", "Bias/fairness testing should be completed before production rollout",
               lambda c: c.bias_testing_completed, "warning"),
    PolicyRule("RAI-05", "Production agents should declare a fallback provider",
               lambda c: c.fallback_provider is not None, "warning"),
]


def evaluate_policy(config: AgentConfig, rules: List[PolicyRule] = POLICY_RULES) -> Dict[str, Any]:
    results = []
    for rule in rules:
        passed = bool(rule.check(config))
        results.append({"rule_id": rule.rule_id, "description": rule.description,
                         "severity": rule.severity, "passed": passed})
    blocking_failures = [r for r in results if not r["passed"] and r["severity"] == "blocking"]
    deployable = len(blocking_failures) == 0
    return {"results": results, "deployable": deployable, "blocking_failures": blocking_failures}


def print_policy_report(config: AgentConfig, outcome: Dict[str, Any]):
    print(f"Policy evaluation for agent: {config.name}")
    print("-" * 60)
    for r in outcome["results"]:
        mark = "PASS" if r["passed"] else ("BLOCK" if r["severity"] == "blocking" else "WARN")
        print(f"  [{mark:5s}] {r['rule_id']} - {r['description']}")
    print("-" * 60)
    verdict = "DEPLOYABLE" if outcome["deployable"] else "NOT DEPLOYABLE (blocking failures present)"
    print(f"  VERDICT: {verdict}")


remittance_agent_cfg = AgentConfig(
    name="remittance-matching-agent",
    has_human_review_for_high_risk=True,
    logs_decisions=True,
    pii_redaction_enabled=False,   # intentionally off to show a blocking failure
    max_autonomy_level="approval_required",
    data_residency="US",
    model_provider="anthropic",
    fallback_provider=None,
    bias_testing_completed=False,
)

outcome = evaluate_policy(remittance_agent_cfg)
print_policy_report(remittance_agent_cfg, outcome)


## 2. PII/PCI redaction guardrail

A demo-grade pattern-based redactor. **Not production-hardened regex** (real systems use
a proper PII detection library or DLP service) — but it shows the shape of a pre-processing
guardrail that should sit in front of any external model call.

In [ ]:
REDACTION_PATTERNS = {
    "CREDIT_CARD": re.compile(r"\b(?:\d[ -]*?){13,16}\b"),
    "SSN": re.compile(r"\b\d{3}-\d{2}-\d{4}\b"),
    "EMAIL": re.compile(r"\b[\w.+-]+@[\w-]+\.[\w.-]+\b"),
    "PHONE": re.compile(r"\b\(?\d{3}\)?[-.\s]?\d{3}[-.\s]?\d{4}\b"),
}


def redact(text: str) -> Dict[str, Any]:
    redacted = text
    findings = []
    for label, pattern in REDACTION_PATTERNS.items():
        for match in pattern.finditer(redacted):
            findings.append({"type": label, "span": match.span()})
        redacted = pattern.sub(f"[REDACTED_{label}]", redacted)
    return {"original_len": len(text), "redacted_text": redacted, "findings_count": len(findings),
            "types_found": sorted(set(f["type"] for f in findings))}


sample = ("Customer John Doe, email john.doe@example.com, phone 415-555-1234, "
          "paid via card 4111 1111 1111 1111 for invoice INV-1001. SSN on file: 123-45-6789.")

result = redact(sample)
print("BEFORE:", sample)
print("\nAFTER :", result["redacted_text"])
n_found, types_found = result["findings_count"], result["types_found"]
print(f"\nFindings: {n_found} across types {types_found}")


## 3. Human-in-the-loop approval gate

Routes a decision to **auto-approve**, **human review**, or **auto-reject** based on a
combination of confidence score and declared risk tier — the standard pattern for putting
a human in the loop only where it matters.

In [ ]:
RISK_TIERS = {"low": 1, "medium": 2, "high": 3}


def route_decision(confidence: float, risk_tier: str, autonomy_level: str) -> Dict[str, str]:
    tier = RISK_TIERS.get(risk_tier, 3)
    if autonomy_level == "advisory":
        return {"decision": "human_review", "reason": "advisory autonomy always requires human sign-off"}
    if tier == 1 and confidence >= 0.90:
        return {"decision": "auto_approve", "reason": "low risk + high confidence"}
    if tier <= 2 and confidence >= 0.97:
        return {"decision": "auto_approve", "reason": "medium risk but very high confidence"}
    if confidence < 0.5:
        return {"decision": "auto_reject", "reason": "confidence too low to act on"}
    return {"decision": "human_review", "reason": "does not meet auto-approve thresholds"}


test_cases = [
    {"confidence": 0.99, "risk_tier": "low", "autonomy_level": "approval_required"},
    {"confidence": 0.995, "risk_tier": "medium", "autonomy_level": "approval_required"},
    {"confidence": 0.80, "risk_tier": "high", "autonomy_level": "approval_required"},
    {"confidence": 0.30, "risk_tier": "low", "autonomy_level": "approval_required"},
    {"confidence": 0.99, "risk_tier": "low", "autonomy_level": "advisory"},
]

for case in test_cases:
    outcome = route_decision(**case)
    decision, reason = outcome["decision"], outcome["reason"]
    print(f"{case} -> {decision:14s} ({reason})")


## 4. Audit trail — append-only decision log

Governance requires an **immutable-style** record of every consequential decision. This
demo uses a hash-chained in-memory log (each entry hashes the previous entry, tamper-evident
style) — the same idea as an append-only ledger, without needing a real database.

In [ ]:
@dataclass
class AuditEntry:
    seq: int
    timestamp: str
    actor: str
    action: str
    details: dict
    prev_hash: str
    this_hash: str = field(init=False)

    def __post_init__(self):
        payload = json.dumps({
            "seq": self.seq, "timestamp": self.timestamp, "actor": self.actor,
            "action": self.action, "details": self.details, "prev_hash": self.prev_hash,
        }, sort_keys=True)
        self.this_hash = hashlib.sha256(payload.encode()).hexdigest()[:16]


class AuditTrail:
    def __init__(self):
        self.entries: List[AuditEntry] = []

    def log(self, actor: str, action: str, details: dict):
        prev_hash = self.entries[-1].this_hash if self.entries else "GENESIS"
        entry = AuditEntry(
            seq=len(self.entries), timestamp=datetime.now(timezone.utc).isoformat(),
            actor=actor, action=action, details=details, prev_hash=prev_hash,
        )
        self.entries.append(entry)
        return entry

    def verify_chain(self) -> bool:
        for i, e in enumerate(self.entries):
            expected_prev = self.entries[i - 1].this_hash if i > 0 else "GENESIS"
            if e.prev_hash != expected_prev:
                return False
        return True

    def print_trail(self):
        for e in self.entries:
            print(f"[{e.seq}] {e.timestamp} | {e.actor:20s} | {e.action:20s} | hash={e.this_hash} prev={e.prev_hash}")


trail = AuditTrail()
trail.log("remittance-matching-agent", "match_suggested", {"invoice": "INV-1003", "confidence": 0.87})
trail.log("human-reviewer:jsmith", "match_approved", {"invoice": "INV-1003"})
trail.log("remittance-matching-agent", "journal_entry_posted", {"invoice": "INV-1003", "erp": "D365"})

trail.print_trail()
print(f"\nChain intact: {trail.verify_chain()}")

# Tamper attempt: mutate an entry's details after the fact, then re-check.
trail.entries[0].details["confidence"] = 0.99   # this does NOT recompute this_hash
print(f"Chain intact after silent tamper attempt: {trail.verify_chain()}  <- hash mismatch would show here if we recomputed vs stored")


---
## 5. Operating Agents at Scale — capacity planner

A simple calculator relating **request volume, agent throughput, and infra/LLM cost** —
the kind of back-of-envelope model used when deciding how many agent instances / what
concurrency limits are needed to operate at scale.

In [ ]:
@dataclass
class CapacityPlan:
    daily_requests: int
    avg_latency_s: float
    peak_multiplier: float          # peak traffic vs. average, e.g. 3x during month-end close
    cost_per_request_usd: float
    target_p95_latency_s: float

    def required_concurrency(self) -> float:
        peak_requests_per_sec = (self.daily_requests * self.peak_multiplier) / (24 * 3600)
        return round(peak_requests_per_sec * self.avg_latency_s, 2)

    def daily_cost_usd(self) -> float:
        return round(self.daily_requests * self.cost_per_request_usd, 2)

    def monthly_cost_usd(self) -> float:
        return round(self.daily_cost_usd() * 30, 2)

    def meets_latency_target(self) -> bool:
        return self.avg_latency_s <= self.target_p95_latency_s

    def report(self) -> str:
        lines = [
            f"Daily requests           : {self.daily_requests:,}",
            f"Peak multiplier          : {self.peak_multiplier}x",
            f"Required concurrency     : ~{self.required_concurrency()} concurrent agent instances",
            f"Est. daily cost          : ${self.daily_cost_usd():,}",
            f"Est. monthly cost        : ${self.monthly_cost_usd():,}",
            f"Meets latency target     : {self.meets_latency_target()}",
        ]
        return "\n".join(lines)


plan = CapacityPlan(
    daily_requests=50_000,
    avg_latency_s=1.2,
    peak_multiplier=3.0,             # month-end close spikes traffic 3x
    cost_per_request_usd=0.004,
    target_p95_latency_s=2.0,
)

print(plan.report())


## 6. Operating Agents at Scale — SLO tracker + error-budget burn

Site-reliability-engineering-style SLO tracking, applied to an agent fleet: define an
availability/accuracy SLO, track actual performance over a rolling window, and compute
how much of the **error budget** has been consumed.

In [ ]:
@dataclass
class SLO:
    name: str
    target: float          # e.g. 0.995 for 99.5%
    window_days: int = 30


@dataclass
class SLOTracker:
    slo: SLO
    total_events: int = 0
    good_events: int = 0

    def record(self, success: bool, count: int = 1):
        self.total_events += count
        if success:
            self.good_events += count

    def current_performance(self) -> float:
        if self.total_events == 0:
            return 1.0
        return round(self.good_events / self.total_events, 5)

    def error_budget_total(self) -> int:
        return int(self.total_events * (1 - self.slo.target))

    def error_budget_consumed(self) -> int:
        return self.total_events - self.good_events

    def error_budget_remaining_pct(self) -> float:
        budget = self.error_budget_total()
        if budget == 0:
            return 100.0 if self.error_budget_consumed() == 0 else 0.0
        remaining = budget - self.error_budget_consumed()
        return round(max(0.0, remaining / budget) * 100, 1)

    def report(self) -> str:
        return (
            f"SLO '{self.slo.name}' target={self.slo.target*100:.2f}% over {self.slo.window_days}d\n"
            f"  Current performance   : {self.current_performance()*100:.3f}%\n"
            f"  Error budget total    : {self.error_budget_total()} bad events allowed\n"
            f"  Error budget consumed : {self.error_budget_consumed()} bad events\n"
            f"  Error budget remaining: {self.error_budget_remaining_pct()}%"
        )


matching_slo = SLOTracker(SLO("matching_accuracy", target=0.995, window_days=30))
# simulate 30 days of volume with a few bad days mixed in
import random
random.seed(11)
for day in range(30):
    daily_volume = random.randint(1000, 2000)
    bad = random.randint(0, 8)
    matching_slo.record(success=True, count=daily_volume - bad)
    matching_slo.record(success=False, count=bad)

print(matching_slo.report())


## 7. Multi-tenant agent registry — governance metadata

At scale you are not operating "an agent," you are operating a **portfolio** of agent
instances across tenants/business units. A registry tracks governance metadata per instance
so audits and reuse reviews don't require spelunking through code repos.

In [ ]:
@dataclass
class AgentInstance:
    instance_id: str
    template_name: str          # which reusable agent template this was built from
    tenant: str
    owner: str
    environment: str            # dev | staging | prod
    maturity_level: float
    last_reviewed: str
    deployed_since: str


class AgentRegistry:
    def __init__(self):
        self.instances: List[AgentInstance] = []

    def register(self, instance: AgentInstance):
        self.instances.append(instance)

    def by_template(self, template_name: str) -> List[AgentInstance]:
        return [i for i in self.instances if i.template_name == template_name]

    def stale_reviews(self, days: int = 90) -> List[AgentInstance]:
        cutoff = datetime.now() - timedelta(days=days)
        return [i for i in self.instances if datetime.fromisoformat(i.last_reviewed) < cutoff]

    def print_summary(self):
        print(f"{'INSTANCE':28s} {'TEMPLATE':22s} {'TENANT':10s} {'ENV':6s} {'MATURITY':8s} LAST REVIEWED")
        for i in self.instances:
            print(f"{i.instance_id:28s} {i.template_name:22s} {i.tenant:10s} {i.environment:6s} "
                  f"{i.maturity_level:<8} {i.last_reviewed}")


registry = AgentRegistry()
registry.register(AgentInstance("remit-match-us-finance", "remittance_matching_v1", "US-Finance",
                                 "AI Lead", "prod", 2.5, "2026-06-01", "2026-01-15"))
registry.register(AgentInstance("remit-match-eu-finance", "remittance_matching_v1", "EU-Finance",
                                 "AI Lead", "prod", 2.0, "2026-02-10", "2026-02-20"))
registry.register(AgentInstance("remit-match-apac-pilot", "remittance_matching_v1", "APAC-Finance",
                                 "Regional Owner", "staging", 1.0, "2026-07-20", "2026-07-01"))

registry.print_summary()

print("\nReuse note: 3 tenants running the SAME template 'remittance_matching_v1' -> good reuse signal.")
print("\nStale reviews (>90 days since last governance review):")
for inst in registry.stale_reviews(days=90):
    print(f"  - {inst.instance_id} (last reviewed {inst.last_reviewed})")


---
### Exercises
1. Add a `RAI-06` rule requiring `data_residency == "EU"` agents to also have `pii_redaction_enabled=True` (data-residency-specific policy).
2. Extend `AuditTrail` with a `recompute_and_compare()` method that detects the tamper attempt in section 4 by recomputing hashes from stored fields.
3. Add a `cost_per_request_usd` breakdown by `risk_tier` to the capacity planner, since human-reviewed decisions cost more (reviewer time) than auto-approved ones.
